# FedMI — Playground Experiments

Run all playground experiments on a **completed training run**:
- **Apply** — evaluate individual client circuits
- **Stitch** — merge client circuits and evaluate
- **Ensemble Distillation** — distill client models into a student
- **LTH Pruning** — iterative magnitude pruning on ensemble model

---

## 1 · Setup

In [ ]:
import os

REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH   = "cvpr"

if not os.path.isdir("FedMI"):
    !git clone -b {BRANCH} {REPO_URL}
os.chdir("FedMI")

from fedmi.env import setup, print_info
setup()
print_info()

## 2 · Select Experiment

Point to a completed training run.

In [ ]:
from fedmi.env import list_experiments, CHECKPOINT_DIR

print("Available experiments:")
list_experiments()

# ── Select your experiment ──
EXP_DIR = os.path.join(CHECKPOINT_DIR, "iid_experiment")  # ← Change to your run folder
print(f"\n✅ Target: {EXP_DIR}")

## 3 · Circuits: Apply & Stitch

In [ ]:
from fedmi.playground import load_config, load_model, load_dataset, load_circuits
from fedmi.playground import eval_full, eval_sufficiency, eval_necessity

cfg = load_config(EXP_DIR)
testloader = load_dataset(cfg)
model = load_model(EXP_DIR, cfg)
circuits_data = load_circuits(EXP_DIR, round_key="last")

print("Evaluating Full Model...")
res = eval_full(model, testloader, cfg)
print(f"Global Acc: {res['overall']:.2f}%")

# CLI Alternative for deep analysis:
# !python playground/circuit_lab.py apply --exp_dir {EXP_DIR} --client_id 0

## 4 · Ensemble & Pruning

In [ ]:
# Run Ensemble Distillation
!python playground/circuit_lab.py ensemble_distill --exp_dir {EXP_DIR} --distill_epochs 5

In [ ]:
# Run LTH Pruning
!python playground/circuit_lab.py lth_prune --exp_dir {EXP_DIR} --prune_iters 5 --finetune_epochs 2

## 5 · View Results & Generated Plots

Display whatever logs and plots were generated by the scripts.

In [ ]:
from fedmi.env import show_images, show_log, list_experiments

# Display generated plots from the experiment dir
show_images(EXP_DIR)

# Show latest log
log_path = os.path.join(EXP_DIR, "logs", "training_log.txt")
show_log(log_path, tail=50)

## 6 · Cleanup

In [ ]:
from fedmi.env import cleanup_checkpoints
cleanup_checkpoints(EXP_DIR, keep_latest=2)